# 4주차 B2 · UCI HAR Inception-ResNet 1D + SE

B1의 Inception-ResNet 1D에 **Squeeze-and-Excitation(SE)** 채널 재가중을 하나만 추가합니다.

이번 실험의 핵심 질문은 다음과 같습니다.

> 같은 Inception-ResNet 구조에서 SE가 학습된 특징 채널의 중요도를 조절하면 더 좋은 UCI HAR 모델이 되는가?

핵심 비교는 `B1(residual on, SE off)`와 `B2(residual on, SE on)`입니다. 두 모델의 데이터 분할, 블록 수, kernel, 학습 조건과 residual scale을 같게 두므로 기본 설정에서 달라지는 것은 SE입니다.

**실행 상태:** 이 파일은 Colab 실행 전 실험 설계본입니다. 저장된 점수를 실제 결과로 간주하지 않습니다.

**중요:** 모델 선택은 피험자 단위 validation으로 수행하고, 공식 test는 최종 구조를 정한 뒤 한 번만 확인합니다.

## 먼저 확인할 결론: SE가 무엇을 선택하는가?

SE는 원본의 가속도 x축이나 자이로스코프 z축을 직접 고르는 장치가 아닙니다. Inception과 Conv1D를 통과한 뒤 만들어진 **학습 특징 채널**의 중요도를 0~1 사이 값으로 조절합니다.

예를 들어 어떤 특징 채널은 반복적인 걷기 진동에, 다른 특징 채널은 비교적 일정한 중력 방향이나 자세에 반응하도록 학습될 수 있습니다. SE는 현재 윈도우 전체를 요약한 뒤 이런 특징 중 어느 것을 더 반영할지 학습합니다.

가설은 다음과 같습니다.

1. Inception이 서로 다른 시간 범위에서 여러 특징 채널을 만듭니다.
2. SE가 각 윈도우에서 더 유용한 학습 특징을 상대적으로 강조하거나 억제합니다.
3. 이 조합이 전체 6-class 성능 또는 일부 행동 구분에 도움이 될 수 있습니다.

하지만 SE는 휴대폰 방향을 직접 보정하지 않고, 시간 위치 정보도 squeeze 과정에서 평균냅니다. 따라서 B2가 B1보다 실제 validation에서 반복적으로 좋아질 때만 효과가 있다고 판단합니다.

## 0. B2 구조와 비교 원칙

B2는 B1의 residual branch에 SE를 추가합니다.

```text
입력 x
 ├─ bottleneck 1×1 → Conv1D(3)
 ├─ bottleneck 1×1 → Conv1D(7)
 ├─ bottleneck 1×1 → Conv1D(15)
 └─ MaxPool(3)      → Conv1D(1)
             ↓ Concatenate
          projection 1×1
             ↓ BatchNormalization
          SE 채널 재가중
             ↓ residual scaling α=0.2
          Add([shortcut, residual])
             ↓ ReLU
```

SE는 `Add` 이전 residual branch에만 둡니다. 이렇게 하면 identity shortcut은 그대로 보존되고, SE가 Inception이 만든 변환 $F(x)$의 특징 채널만 조절합니다. `Add` 이후에 SE를 두면 shortcut까지 함께 재가중되어 B1과의 차이를 해석하기 어려워집니다.

### B1과 B2에서 고정하는 조건

| 항목 | B1 | B2 |
|---|---|---|
| Inception kernel | 3·7·15 | 3·7·15 |
| 블록 | 32채널 2개 + 64채널 2개 | 동일 |
| Residual | 사용 | 사용 |
| Residual scale | 0.2 | 0.2 |
| SE | 사용하지 않음 | 사용 |
| 학습·검증 분리와 학습 조건 | 동일 | 동일 |

따라서 기본 설정의 `B2 - B1` 차이는 SE를 추가한 효과로 해석할 수 있습니다. 다만 여러 모델을 같은 validation으로 반복 선택하면 validation에도 점차 맞춰질 수 있으므로, 비교 기준을 먼저 정하고 공식 test는 보지 않습니다.

## 1. 라이브러리 불러오기

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

## 2. 실험 설정

먼저 `USE_RESIDUAL=True`, `USE_SE=True`를 유지하고 `SEED`만 0·1·2로 바꿉니다. 이것이 B2의 정식 결과입니다.

`SE_RATIO=8`은 32채널을 4개, 64채널을 8개의 중간 값으로 압축합니다. ratio 16은 이 작은 모델에서 각각 2개·4개까지 줄어 지나치게 좁을 수 있어 첫 실험은 8로 시작합니다. 이 값도 정답이 아니라 비교를 위한 시작값입니다.

In [ ]:
SEED = 0
USE_RESIDUAL = True
USE_SE = True
SE_RATIO = 8
RESIDUAL_SCALE = 0.2

np.random.seed(SEED)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

EXPERIMENT_NAME = (
    f"B2-res{int(USE_RESIDUAL)}"
    f"-se{int(USE_SE)}"
    f"-r{SE_RATIO}"
)
print("experiment:", EXPERIMENT_NAME, "| seed:", SEED)

## 3. Google Drive와 데이터 경로 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍/2week/DATA/UCI-HAR Dataset"
assert os.path.exists(DATA_PATH), f"데이터 경로를 확인하세요: {DATA_PATH}"

## 4. 9개 센서 신호와 데이터 로더 정의

In [ ]:
signals = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]


def load_data(split):
    channels = []

    for signal in signals:
        path = f"{DATA_PATH}/{split}/Inertial Signals/{signal}_{split}.txt"
        channels.append(np.loadtxt(path))

    # (9, N, 128)이 아니라 각 (N, 128) 배열을 마지막 축에 쌓아 (N, 128, 9)로 만듭니다.
    X = np.stack(channels, axis=-1)
    y = np.loadtxt(f"{DATA_PATH}/{split}/y_{split}.txt").astype(int) - 1
    subject = np.loadtxt(f"{DATA_PATH}/{split}/subject_{split}.txt").astype(int)

    return X, y, subject


X_train_all, y_train_all, s_train_all = load_data("train")
X_test, y_test, s_test = load_data("test")

print("official train:", X_train_all.shape)
print("official test :", X_test.shape, "(최종 셀 전까지 평가하지 않음)")

## 5. 피험자 단위 validation 분리

같은 사람의 윈도우가 학습과 validation에 동시에 들어가지 않도록 피험자 5·11·17·23·29를 통째로 분리합니다. 특히 5번과 17번의 `SITTING`/`STANDING` 성능을 핵심 지표로 봅니다.

In [ ]:
VAL_SUBJECTS = [5, 11, 17, 23, 29]
FOCUS_SUBJECTS = [5, 17]

val_mask = np.isin(s_train_all, VAL_SUBJECTS)

X_val = X_train_all[val_mask]
y_val = y_train_all[val_mask]
s_val = s_train_all[val_mask]

X_train = X_train_all[~val_mask]
y_train = y_train_all[~val_mask]
s_train = s_train_all[~val_mask]

print("train:", X_train.shape, "subjects:", sorted(np.unique(s_train)))
print("val  :", X_val.shape, "subjects:", sorted(np.unique(s_val)))

## 6. 학습 데이터 통계로 정규화

In [ ]:
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = ((X_train - mean) / (std + 1e-8)).astype("float32")
X_val = ((X_val - mean) / (std + 1e-8)).astype("float32")
X_test = ((X_test - mean) / (std + 1e-8)).astype("float32")

n_outputs = 6
y_train_onehot = to_categorical(y_train, num_classes=n_outputs)
y_val_onehot = to_categorical(y_val, num_classes=n_outputs)

## 7. Inception-residual 1D 블록과 SE 위치

Inception branch와 projection은 B1과 동일합니다. B2의 유일한 구조 변경은 `BatchNormalization → SE → residual scaling → Add`입니다.

- SE를 각 Inception branch에 따로 넣지 않습니다. 그러면 구조와 파라미터 변화가 커져 단순 비교가 어려워집니다.
- SE를 shortcut에는 넣지 않습니다. 입력 정보가 그대로 흐르는 residual 통로를 보존합니다.
- SE의 sigmoid gate는 초기에 대체로 0.5 근처일 수 있어 B1보다 residual 신호를 더 줄일 수도 있습니다. 따라서 향상을 미리 가정하지 않습니다.

### 7-1. 1D Squeeze-and-Excitation 블록

입력 shape이 `(batch, time, channels)`일 때 동작은 다음과 같습니다.

1. **Squeeze:** `GlobalAveragePooling1D`로 시간축을 평균내어 채널마다 하나의 값을 만듭니다.
2. **Excitation:** 작은 Dense 층 두 개로 채널 간 관계를 학습하고 sigmoid 중요도를 만듭니다.
3. **Scale:** 중요도를 원래 특징맵의 각 시간 위치에 같은 값으로 곱합니다.

$$z_c=\frac{1}{T}\sum_{t=1}^{T}U_{t,c},\qquad s=\sigma(W_2\operatorname{ReLU}(W_1z))$$

예를 들어 `(batch, 128, 32)`는 squeeze 후 `(batch, 32)`, excitation 중간에는 `(batch, 4)`, 다시 `(batch, 32)`가 된 뒤 `(batch, 1, 32)`로 바뀌어 원래 특징에 곱해집니다.

In [ ]:
def se_block_1d(x, reduction_ratio=8, name="se"):
    channels = x.shape[-1]
    if channels is None:
        raise ValueError("SE 블록은 채널 수가 확정되어 있어야 합니다.")

    channels = int(channels)
    hidden = max(channels // reduction_ratio, 1)

    # Squeeze: 각 특징 채널의 시간축 전체 반응을 하나의 값으로 요약합니다.
    scale = layers.GlobalAveragePooling1D(name=f"{name}_squeeze")(x)

    # Excitation: 채널 간 관계를 이용해 0~1 중요도를 학습합니다.
    scale = layers.Dense(hidden, activation="relu", name=f"{name}_reduce")(scale)
    scale = layers.Dense(channels, activation="sigmoid", name=f"{name}_expand")(scale)
    scale = layers.Reshape((1, channels), name=f"{name}_reshape")(scale)

    # Scale: 모든 시간 위치에서 같은 채널 중요도를 곱합니다.
    return layers.Multiply(name=f"{name}_scale")([x, scale])

In [ ]:
def inception_residual_block(
    x,
    filters,
    bottleneck_filters=16,
    kernel_sizes=(3, 7, 15),
    residual_scale=0.2,
    use_residual=True,
    use_se=True,
    se_ratio=8,
    name="ir",
):
    if filters % 4 != 0:
        raise ValueError("filters는 네 branch에 나눌 수 있도록 4의 배수여야 합니다.")

    branch_filters = filters // 4
    shortcut = x

    # 큰 kernel을 적용하기 전에 채널을 줄여 계산량을 제한합니다.
    bottleneck = layers.Conv1D(
        bottleneck_filters, 1, padding="same", use_bias=False,
        name=f"{name}_bottleneck",
    )(x)
    bottleneck = layers.BatchNormalization(name=f"{name}_bottleneck_bn")(bottleneck)
    bottleneck = layers.Activation("relu", name=f"{name}_bottleneck_relu")(bottleneck)

    branches = []
    for kernel_size in kernel_sizes:
        branch = layers.Conv1D(
            branch_filters, kernel_size, padding="same", use_bias=False,
            name=f"{name}_conv_k{kernel_size}",
        )(bottleneck)
        branches.append(branch)

    pool_branch = layers.MaxPooling1D(
        pool_size=3, strides=1, padding="same", name=f"{name}_pool"
    )(x)
    pool_branch = layers.Conv1D(
        branch_filters, 1, padding="same", use_bias=False,
        name=f"{name}_pool_projection",
    )(pool_branch)
    branches.append(pool_branch)

    merged = layers.Concatenate(axis=-1, name=f"{name}_concatenate")(branches)

    # Inception 결과를 shortcut과 같은 채널 수로 투영합니다.
    residual = layers.Conv1D(
        filters, 1, padding="same", use_bias=False,
        name=f"{name}_residual_projection",
    )(merged)
    residual = layers.BatchNormalization(name=f"{name}_residual_bn")(residual)

    # B2 변경: shortcut은 그대로 두고 residual branch의 특징만 재가중합니다.
    if use_se:
        residual = se_block_1d(
            residual, reduction_ratio=se_ratio, name=f"{name}_se"
        )

    # 모든 ablation 조건에 같은 residual 배율을 적용합니다.
    # 비교 조건에서 shortcut 유무만 달라지게 합니다.
    residual = layers.Rescaling(
        residual_scale, name=f"{name}_residual_scale"
    )(residual)

    if use_residual:
        if x.shape[-1] != filters:
            shortcut = layers.Conv1D(
                filters, 1, padding="same", use_bias=False,
                name=f"{name}_shortcut_projection",
            )(shortcut)
            shortcut = layers.BatchNormalization(name=f"{name}_shortcut_bn")(shortcut)

        x = layers.Add(name=f"{name}_add")([shortcut, residual])
    else:
        # shortcut만 뺀 대조 실험입니다. 나머지 Inception 경로는 동일합니다.
        x = residual

    return layers.Activation("relu", name=f"{name}_out")(x)

## 8. B2 모델 만들기

Stem, 네 Inception-residual 블록, pooling, 분류 head는 B1과 같습니다. 각 residual branch에만 SE를 추가합니다. 따라서 `model.count_params()`가 B1보다 얼마나 늘었는지도 함께 확인합니다.

In [ ]:
def build_b2_model(use_residual=True, use_se=True, se_ratio=8, residual_scale=0.2):
    inputs = layers.Input(shape=(128, 9), name="sensor_window")

    x = layers.Conv1D(32, 5, padding="same", use_bias=False, name="stem_conv")(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("relu", name="stem_relu")(x)

    x = inception_residual_block(
        x, filters=32, bottleneck_filters=16,
        residual_scale=residual_scale, use_residual=use_residual, use_se=use_se, se_ratio=se_ratio, name="ir32_1",
    )
    x = inception_residual_block(
        x, filters=32, bottleneck_filters=16,
        residual_scale=residual_scale, use_residual=use_residual, use_se=use_se, se_ratio=se_ratio, name="ir32_2",
    )

    x = layers.MaxPooling1D(2, name="downsample_pool")(x)
    x = layers.Conv1D(64, 1, padding="same", use_bias=False, name="channel_up")(x)
    x = layers.BatchNormalization(name="channel_up_bn")(x)
    x = layers.Activation("relu", name="channel_up_relu")(x)

    x = inception_residual_block(
        x, filters=64, bottleneck_filters=24,
        residual_scale=residual_scale, use_residual=use_residual, use_se=use_se, se_ratio=se_ratio, name="ir64_1",
    )
    x = inception_residual_block(
        x, filters=64, bottleneck_filters=24,
        residual_scale=residual_scale, use_residual=use_residual, use_se=use_se, se_ratio=se_ratio, name="ir64_2",
    )

    x = layers.GlobalAveragePooling1D(name="global_average_pooling")(x)
    x = layers.Dropout(0.4, name="classifier_dropout")(x)
    outputs = layers.Dense(n_outputs, activation="softmax", name="activity")(x)

    return Model(inputs, outputs, name="UCI_HAR_B2_Inception_ResNet1D_SE")


model = build_b2_model(
    use_residual=USE_RESIDUAL,
    use_se=USE_SE,
    se_ratio=SE_RATIO,
    residual_scale=RESIDUAL_SCALE,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()
print(f"parameters: {model.count_params():,}")

## 9. `Concatenate`와 `Add` shape 확인

`Concatenate` 뒤에는 여러 branch의 채널이 모이고, projection을 거친 residual과 shortcut은 같은 shape이어야 합니다. 이 검사는 잘못된 `Add` 설계를 실행 전에 발견하기 위한 것입니다.

B2에서는 SE 전후 shape도 같아야 합니다. SE는 값을 조절하지만 시간 길이와 채널 수를 바꾸지 않습니다.

In [ ]:
for block_name in ["ir32_1", "ir32_2", "ir64_1", "ir64_2"]:
    merged_shape = model.get_layer(f"{block_name}_concatenate").output.shape
    residual_shape = model.get_layer(f"{block_name}_residual_bn").output.shape
    output_shape = model.get_layer(f"{block_name}_out").output.shape
    print(block_name, "concat:", merged_shape, "residual:", residual_shape, "out:", output_shape)

    if USE_RESIDUAL:
        add_layer = model.get_layer(f"{block_name}_add")
        left_shape, right_shape = [tensor.shape for tensor in add_layer.input]
        assert left_shape == right_shape, f"{block_name} Add shape 불일치"

    if USE_SE:
        se_input_shape = model.get_layer(f"{block_name}_residual_bn").output.shape
        se_output_shape = model.get_layer(f"{block_name}_se_scale").output.shape
        assert se_input_shape == se_output_shape, f"{block_name} SE shape 불일치"
        print(" " * 10, "SE in/out:", se_input_shape, "->", se_output_shape)


## 10. 학습

기존 S0와 마찬가지로 최대 100에폭, batch size 64, Adam과 EarlyStopping을 사용합니다. 구조 외의 조건을 가능한 한 고정하고 학습 시간도 함께 기록합니다.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=10,
        restore_best_weights=True, verbose=1,
    )
]

train_start = time.perf_counter()

history = model.fit(
    X_train,
    y_train_onehot,
    validation_data=(X_val, y_val_onehot),
    epochs=100,
    batch_size=64,
    shuffle=True,
    callbacks=callbacks,
    verbose=1,
)

training_seconds = time.perf_counter() - train_start
best_epoch = int(np.argmin(history.history["val_loss"])) + 1

print(f"training time: {training_seconds:.1f} sec")
print(f"best epoch by val_loss: {best_epoch}")

## 11. Validation 전체 평가

In [ ]:
class_names = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING",
]

y_val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
cm_val = confusion_matrix(y_val, y_val_pred, labels=range(n_outputs))

print(f"validation accuracy : {val_accuracy:.4f}")
print(f"validation macro F1 : {val_f1:.4f}")
print(classification_report(
    y_val, y_val_pred, target_names=class_names,
    digits=4, zero_division=0,
))
print("confusion matrix")
print(cm_val)

## 12. `SITTING`/`STANDING`과 피험자별 평가

전체 정확도가 좋아져도 기존 핵심 문제인 앉음↔섬 오류가 늘어날 수 있으므로 별도로 기록합니다.

In [ ]:
SIT, STAND = 3, 4

print("subject | all acc | SIT/STAND acc | SIT->STAND | STAND->SIT")
for subject in VAL_SUBJECTS:
    mask = s_val == subject
    ss_mask = mask & np.isin(y_val, [SIT, STAND])
    sit_to_stand = np.sum(mask & (y_val == SIT) & (y_val_pred == STAND))
    stand_to_sit = np.sum(mask & (y_val == STAND) & (y_val_pred == SIT))
    mark = " <- focus" if subject in FOCUS_SUBJECTS else ""

    print(
        f"{subject:7d} | {accuracy_score(y_val[mask], y_val_pred[mask]):.4f}  | "
        f"{accuracy_score(y_val[ss_mask], y_val_pred[ss_mask]):.4f}        | "
        f"{sit_to_stand:10d} | {stand_to_sit:10d}{mark}"
    )

focus_mask = np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])
other_mask = ~np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

focus_ss_acc = accuracy_score(y_val[focus_mask], y_val_pred[focus_mask])
other_ss_acc = accuracy_score(y_val[other_mask], y_val_pred[other_mask])
val_ss_err = int(cm_val[SIT, STAND] + cm_val[STAND, SIT])
val_dynamic_err = int(cm_val[0:3, 0:3].sum() - np.trace(cm_val[0:3, 0:3]))

## 13. 추론 시간 측정

GPU·CPU 종류에 따라 절대 시간은 달라집니다. 따라서 같은 Colab 런타임에서 다른 모델과 측정한 값끼리만 비교합니다.

In [ ]:
timing_batch = tf.convert_to_tensor(X_val[:256])

# 초기 그래프 준비와 장치 준비 시간을 제외하기 위한 warm-up입니다.
for _ in range(3):
    model(timing_batch, training=False).numpy()

repeats = 20
inference_start = time.perf_counter()
for _ in range(repeats):
    model(timing_batch, training=False).numpy()
inference_seconds = time.perf_counter() - inference_start

inference_ms_per_sample = (
    inference_seconds * 1000 / (repeats * len(timing_batch))
)
print(f"inference: {inference_ms_per_sample:.4f} ms/sample")

## 14. Gradient norm 관찰

고정된 validation mini-batch에서 loss를 한 번 계산하고 각 Conv1D kernel의 gradient norm을 출력합니다. 앞쪽 층의 gradient가 반복적으로 매우 작고 residual을 끈 실험보다도 작다면 gradient 전달 문제를 의심할 수 있습니다.

단, 한 시점의 gradient norm만으로 gradient 소실을 증명할 수는 없습니다. 동일 seed에서 `USE_RESIDUAL=True/False`를 비교하고 학습곡선·수렴 에폭·정확도를 함께 봅니다.

B2에서는 SE가 residual branch의 크기 자체를 바꾸므로 B1과 gradient norm이 달라질 수 있습니다. 이 차이를 곧바로 더 좋은 gradient 전달이라고 해석하지 않고 성능과 수렴 양상을 함께 봅니다.


In [ ]:
probe_x = tf.convert_to_tensor(X_val[:64])
probe_y = tf.convert_to_tensor(y_val_onehot[:64])

with tf.GradientTape() as tape:
    probe_prediction = model(probe_x, training=False)
    probe_loss = tf.reduce_mean(
        tf.keras.losses.categorical_crossentropy(probe_y, probe_prediction)
    )

gradients = tape.gradient(probe_loss, model.trainable_variables)
gradient_rows = []

for variable, gradient in zip(model.trainable_variables, gradients):
    variable_path = getattr(variable, "path", variable.name)
    if gradient is not None and "kernel" in variable_path:
        gradient_rows.append((variable_path, float(tf.norm(gradient))))

print("loss:", float(probe_loss))
print("\n앞쪽 kernel gradient norm")
for name, norm in gradient_rows[:6]:
    print(f"{name:55s} {norm:.8f}")

print("\n뒤쪽 kernel gradient norm")
for name, norm in gradient_rows[-6:]:
    print(f"{name:55s} {norm:.8f}")

## 15. 학습곡선

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} loss · seed {SEED}")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} accuracy · seed {SEED}")
plt.legend()

plt.tight_layout()
plt.show()

## 16. 결과 한 줄 요약

In [ ]:
print(
    f"{EXPERIMENT_NAME} | seed {SEED} | "
    f"val acc {val_accuracy:.4f} | val F1 {val_f1:.4f} | "
    f"focus SIT/STAND(5,17) {focus_ss_acc:.4f} | "
    f"other SIT/STAND(11,23,29) {other_ss_acc:.4f} | "
    f"SIT<->STAND errors {val_ss_err} | dynamic errors {val_dynamic_err} | "
    f"epochs {len(history.history['val_loss'])} (best {best_epoch}) | "
    f"params {model.count_params():,} | train sec {training_seconds:.1f} | "
    f"inference ms/sample {inference_ms_per_sample:.4f}"
)

## 17. B1과 B2 결과 기록표

B1과 B2를 같은 seed끼리 비교합니다. 시간 제약이 있다면 우선 아래 여섯 번만 실행합니다.

| 실험 | seed | SE | Val Accuracy | Macro F1 | Focus SIT/STAND | Params | Best epoch | Train sec | ms/sample |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| B1 | 0 | Off |  |  |  |  |  |  |  |
| B2 | 0 | On |  |  |  |  |  |  |  |
| B1 | 1 | Off |  |  |  |  |  |  |  |
| B2 | 1 | On |  |  |  |  |  |  |  |
| B1 | 2 | Off |  |  |  |  |  |  |  |
| B2 | 2 | On |  |  |  |  |  |  |  |

### 해석 기준

- 세 seed 평균과 표준편차를 보고, 같은 seed에서 `B2 - B1` 방향이 반복되는지 확인합니다.
- Accuracy만 보지 않고 Macro F1, 피험자별 성능과 행동별 오류를 같이 봅니다.
- B2의 추가 파라미터와 추론 시간에 비해 개선 폭이 충분한지도 확인합니다.
- SE가 좋아도 “원본 중력 센서축을 선택했다”고 결론 내리지 않습니다. 학습 특징 채널의 재가중이 도움이 되었을 가능성까지만 말합니다.
- B2가 전체 성능을 높이더라도 특정 행동이나 최저 성능 피험자를 악화시키면 그 trade-off를 기록합니다.

## 18. 🔒 공식 test 평가

모든 후보를 같은 validation 조건으로 비교하고 최종 모델을 결정한 뒤 한 번만 실행합니다. 기본값에서는 오류를 발생시키지 않고 “건너뜀” 메시지만 출력하므로, Colab의 “모두 실행” 결과를 깨끗하게 저장할 수 있습니다.

In [ ]:
FINAL_TEST = False

if not FINAL_TEST:
    print(
        "공식 test를 건너뜁니다. validation으로 최종 모델을 정한 뒤 "
        "이 셀의 FINAL_TEST=True로 변경해 한 번만 실행하세요."
    )
else:
    y_test_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
    cm_test = confusion_matrix(y_test, y_test_pred, labels=range(n_outputs))

    print(f"test accuracy : {test_accuracy:.4f}")
    print(f"test macro F1 : {test_f1:.4f}")
    print(classification_report(
        y_test, y_test_pred, target_names=class_names,
        digits=4, zero_division=0,
    ))
    print("confusion matrix")
    print(cm_test)

    test_ss_err = int(cm_test[SIT, STAND] + cm_test[STAND, SIT])
    test_dynamic_err = int(cm_test[0:3, 0:3].sum() - np.trace(cm_test[0:3, 0:3]))

    print(
        f"{EXPERIMENT_NAME} TEST | seed {SEED} | acc {test_accuracy:.4f} | "
        f"macro F1 {test_f1:.4f} | SIT<->STAND {test_ss_err} | "
        f"dynamic errors {test_dynamic_err}"
    )

## 19. 참고 자료

- Szegedy et al., *Inception-v4, Inception-ResNet and the Impact of Residual Connections on Learning*, 2016
- He et al., *Deep Residual Learning for Image Recognition*, 2015
- Hu et al., *Squeeze-and-Excitation Networks*, 2017
- Ismail Fawaz et al., *InceptionTime: Finding AlexNet for Time Series Classification*, 2019
- UCI Machine Learning Repository, *Human Activity Recognition Using Smartphones*